# Laboratorio ETL: CSV → Python → SQL Server
**Curso:** Base de Datos Avanzadas y Big Data · **Unidad II:** Business Intelligence · **Semana 7**  
**Saber esencial:** Diseño de procesos ETL · **Duración sugerida:** 2 horas (con BD preparada)  
**Logro de la unidad:** diseñar soluciones de inteligencia de negocios que respondan a necesidades identificadas, utilizando metodologías y herramientas adecuadas.

**Escenario:** Una empresa comercial dispone de `ventas.csv` (500 líneas sintéticas). La gerencia necesita analizar ventas por producto, cliente/ciudad y tiempo. Debe cargarse un Data Mart en SQL Server con `DimCliente`, `DimProducto`, `DimFecha` y `FactVentas`. La ciudad es atributo del cliente.

**Granularidad:** una fila válida de la tabla de hechos equivale a una línea de venta identificada por `IdVenta`.

**VERSIÓN DOCENTE — SOLUCIÓN DE REFERENCIA**. No distribuir con la versión del estudiante.

**Material previo:** `ventas.csv`, `Metadata_Ventas.docx`, `Modelo_Dimensional_Ventas_Actualizado.docx` y `DataMartVentas.sql`. Las cuatro tablas deben crearse con el script SQL antes de la carga; no precargue registros.

**Versión:** 1.1 · **Entorno recomendado:** Anaconda `etl_bi`, kernel `Python (ETL BI)`.

## Propósito pedagógico y criterios de corrección
Evalúa diagnóstico de calidad, toma de decisiones justificadas, correspondencia fuente→destino, carga transaccional, trazabilidad, reejecución y conciliación. Distintas políticas de cuarentena/corrección son admisibles si preservan integridad y consistencia.

**Política elegida en esta referencia:**
1. Todo `IdVenta` repetido se pone en cuarentena (ambas líneas) porque no puede saberse cuál es verdadera.
2. Fechas ISO (`AAAA-MM-DD`) o `DD/MM/AAAA` se interpretan explícitamente; fechas imposibles se rechazan.
3. Nombres, ciudad y categoría se normalizan; para cada clave válida se adopta el valor modal **solo con evidencia consistente**. Los faltantes se completan únicamente desde esa clave observada en filas confiables.
4. Los códigos de producto cuya descripción no coincide con el catálogo consolidado se ponen en cuarentena; no se infiere una clave desconocida por similitud.
5. Precios y cantidades no positivos/no numéricos se rechazan; no se sustituyen por cero.
6. La carga es idempotente para el mismo CSV: dimensiones faltantes y nuevos hechos se insertan, existentes se omiten. No actualizamos silenciosamente valores de dimensiones ya persistidos; las divergencias con el destino generan error para revisión (laboratorio introductorio).

**Limitación:** la resolución de atributos por moda supone estabilidad descriptiva para esta fuente sintética (SCD 1 simplificado). Para producción requiere catálogo maestro y reglas de cambio histórico.

## Antes del laboratorio: guía de configuración (obligatoria)

**Documento independiente:** `Guia_Entorno_ETL_Anaconda.docx`. Siga esa guía **en Anaconda Prompt, antes de abrir este cuaderno**, para crear y activar `etl_bi`, instalar dependencias, registrar el kernel y comprobar el controlador ODBC. **No ejecute comandos `conda create` dentro de las celdas del notebook.**

1. En Anaconda Prompt active `conda activate etl_bi` e inicie `jupyter lab` (o Jupyter Notebook, si lo prefiere).
2. En Jupyter seleccione el kernel **Python (ETL BI)**.
3. Compruebe que `ventas.csv` existe en `CSV_PATH` y que `DataMartVentas.sql` se ejecutó previamente en SSMS.
4. Ajuste `SERVIDOR` y `CONTROLADOR` a su máquina. El ejemplo utiliza autenticación integrada de Windows. No incluya contraseñas en el notebook.
5. Ejecute las verificaciones siguientes antes de pasar a los desafíos.

**Nota de seguridad:** `TrustServerCertificate=yes` solo se usa en este ejemplo de práctica local. En producción, configure certificados confiables y la verificación TLS correspondiente.

In [1]:
# Diagnóstico del entorno: no modifica SQL Server.
import sys
from pathlib import Path
import pandas as pd
import sqlalchemy
import pyodbc

print("Python:", sys.version.split()[0])
print("Ejecutable de Python:", sys.executable)
print("pandas:", pd.__version__)
print("SQLAlchemy:", sqlalchemy.__version__)
print("pyodbc:", pyodbc.version)
print("Controladores ODBC:", pyodbc.drivers())
print("Kernel esperado: Python (ETL BI); confirme además su selección en Jupyter.")


Python: 3.11.17
Ejecutable de Python: C:\Users\Gustavo Coronel\.conda\envs\etl_bi\python.exe
pandas: 3.0.6
SQLAlchemy: 2.0.52
pyodbc: 5.3.0
Controladores ODBC: ['SQL Server', 'SQL Server Native Client 11.0', 'Oracle en OraDB21Home1', 'BMF SQLite3 Driver', 'ODBC Driver 17 for SQL Server', 'ODBC Driver 18 for SQL Server', 'Oracle en OraDB19Home1', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']
Kernel esperado: Python (ETL BI); confirme además su selección en Jupyter.


### Comprobación de archivos y preparación de la conexión

Ubique `ventas.csv` en la carpeta de trabajo o actualice `CSV_PATH`. El código siguiente **configura** el motor SQLAlchemy, pero **no realiza todavía una conexión a SQL Server**. Confirme que dispone del controlador ODBC seleccionado.

In [4]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

CSV_PATH = Path("D:\\Alfa\\ventas.csv")  # Actualizar según la carpeta
SERVIDOR = r"localhost"  # Ajustar a su instancia
BASE_DATOS = "DataMartVentas"
CONTROLADOR = "ODBC Driver 18 for SQL Server"  # Revisar con pyodbc.drivers()
# Con Driver 18, TrustServerCertificate=yes facilita prácticas con certificados locales.
# Solo para desarrollo local; en producción configure certificados TLS válidos.
CADENA_ODBC = (
    f"DRIVER={{{CONTROLADOR}}};SERVER={SERVIDOR};DATABASE={BASE_DATOS};"
    "Trusted_Connection=yes;Encrypt=yes;TrustServerCertificate=yes;"
)
engine = create_engine(URL.create("mssql+pyodbc", query={"odbc_connect": CADENA_ODBC}), future=True)
print("Ruta CSV:", CSV_PATH.resolve())
print("Drivers ODBC disponibles:", pyodbc.drivers())
if not CSV_PATH.is_file():
    raise FileNotFoundError(f"No se encontró el archivo CSV: {CSV_PATH.resolve()}")
if CONTROLADOR not in pyodbc.drivers():
    raise RuntimeError(f"No está disponible {CONTROLADOR}. Controladores: {pyodbc.drivers()}")
print("Archivo CSV encontrado y controlador ODBC disponible.")

Ruta CSV: D:\Alfa\ventas.csv
Drivers ODBC disponibles: ['SQL Server', 'SQL Server Native Client 11.0', 'Oracle en OraDB21Home1', 'BMF SQLite3 Driver', 'ODBC Driver 17 for SQL Server', 'ODBC Driver 18 for SQL Server', 'Oracle en OraDB19Home1', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']
Archivo CSV encontrado y controlador ODBC disponible.


### Prueba de conectividad y estructura de destino

Esta celda efectúa una conexión de **solo lectura**. La base de datos y sus cuatro tablas se deben haber creado con el script `DataMartVentas.sql` en SSMS. Si falla, revise el nombre de la instancia, autenticación, servicio SQL Server y controlador ODBC antes de continuar.

In [5]:
tablas_requeridas = {"DimCliente", "DimProducto", "DimFecha", "FactVentas"}
with engine.connect() as conexion:
    nombre_bd = conexion.execute(text("SELECT DB_NAME()")).scalar_one()
    tablas_encontradas = set(pd.read_sql_query(
        text("SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA = 'dbo' AND TABLE_TYPE = 'BASE TABLE'"),
        conexion
    )["TABLE_NAME"])
print("Base de datos:", nombre_bd)
print("Tablas esperadas:", sorted(tablas_requeridas))
print("Tablas encontradas:", sorted(tablas_encontradas & tablas_requeridas))
assert nombre_bd == BASE_DATOS, "La conexión apunta a otra base de datos"
assert tablas_requeridas <= tablas_encontradas, "Faltan tablas: " + str(tablas_requeridas - tablas_encontradas)
print("Conexión y estructura verificadas (solo lectura).")

Base de datos: DataMartVentas
Tablas esperadas: ['DimCliente', 'DimFecha', 'DimProducto', 'FactVentas']
Tablas encontradas: ['DimCliente', 'DimFecha', 'DimProducto', 'FactVentas']
Conexión y estructura verificadas (solo lectura).


## Solución 1. Extracción y perfilado
Se conserva `dtype=str` para revisar la fuente sin coerción automática.

In [7]:
df = pd.read_csv(CSV_PATH, encoding="utf-8-sig", dtype=str, keep_default_na=False)
columnas = ["IdVenta", "Fecha", "IdCliente", "Cliente", "Ciudad", "IdProducto", "Producto", "Categoria", "Cantidad", "PrecioUnitario"]
assert list(df.columns) == columnas, "Columnas distintas de la metadata entregada"
assert len(df) == 500, "Se esperaba el CSV original de 500 filas"
print("Filas:", len(df), "Columnas:", len(df.columns))
print("Identificadores IdVenta repetidos:", df.IdVenta.str.strip().duplicated(keep=False).sum(), "filas involucradas")
display(df.head())

Filas: 500 Columnas: 10
Identificadores IdVenta repetidos: 4 filas involucradas


,IdVenta,Fecha,IdCliente,Cliente,Ciudad,IdProducto,Producto,Categoria,Cantidad,PrecioUnitario
0,V0001,2026-07-09,C007,Elena Flores,Chiclayo,P006,Disco SSD 1TB,Almacenamiento,4,336.00
1,V0002,2026-07-29,C018,José Mendoza,Puno,P002,Mouse Logitech,Accesorios,1,78.75
2,V0003,2026-09-12,C009,Lucía Campos,Ica,P005,Impresora Epson,Tecnología,2,714.00
3,V0004,2026-07-31,C017,Carmen Ruiz,Lima,P001,Laptop Lenovo,Tecnología,2,2327.50
4,V0005,2026-07-29,C003,Rosa Díaz,Trujillo,P010,Webcam HD,Accesorios,5,120.00


## Solución 2. Normalización, reglas y cuarentena
El reporte contiene todas las filas (aceptadas o rechazadas) con `NumeroFilaCSV`; así se preserva la trazabilidad incluso si `IdVenta` está duplicado.

In [8]:
import re
from decimal import Decimal, InvalidOperation
from collections import Counter

def texto(v):
    return " ".join(str(v).strip().split())

def firma(v):
    return texto(v).casefold()

def numero_positivo(v, entero=False):
    try:
        n = Decimal(texto(v))
        if not n.is_finite() or n <= 0 or (entero and n != n.to_integral_value()):
            return None
        if n.as_tuple().exponent < -2 and not entero:
            return None   # la fuente declara precios con dos decimales
        return int(n) if entero else n
    except (InvalidOperation, ValueError):
        return None

def fecha_valida(v):
    s = texto(v)
    formato = "%Y-%m-%d" if re.fullmatch(r"\d{4}-\d{2}-\d{2}", s) else (
              "%d/%m/%Y" if re.fullmatch(r"\d{2}/\d{2}/\d{4}", s) else None)
    if formato is None:
        return pd.NaT
    return pd.to_datetime(s, format=formato, errors="coerce")

trabajo = df.copy()
trabajo.insert(0, "NumeroFilaCSV", range(2, len(trabajo) + 2))  # incluye encabezado
for c in columnas:
    trabajo[c] = trabajo[c].map(texto)
trabajo["FechaConvertida"] = trabajo.Fecha.map(fecha_valida)
trabajo["CantidadConvertida"] = trabajo.Cantidad.map(lambda v: numero_positivo(v, entero=True))
trabajo["PrecioConvertido"] = trabajo.PrecioUnitario.map(numero_positivo)

def catalogo_confiable(tabla, clave, campo, patron):
    """Valor modal de descripciones de claves con patrón esperado; omite celdas vacías."""
    subconjunto = tabla.loc[tabla[clave].str.fullmatch(patron).fillna(False), [clave, campo]].copy()
    subconjunto = subconjunto[subconjunto[campo].ne("")]
    resultado = {}
    for codigo, grupo in subconjunto.groupby(clave):
        valores = grupo[campo].tolist()
        cuentas = Counter(firma(x) for x in valores)
        firma_modal, _ = sorted(cuentas.items(), key=lambda z: (-z[1], z[0]))[0]
        originales = [x for x in valores if firma(x) == firma_modal]
        resultado[codigo] = Counter(originales).most_common(1)[0][0]
    return resultado

patron_cliente = r"C\d{3}"
patron_producto = r"P0\d{2}"  # catálogo operativo P001..P012; P999 no se acepta
cat = {
    "Cliente": catalogo_confiable(trabajo, "IdCliente", "Cliente", patron_cliente),
    "Ciudad": catalogo_confiable(trabajo, "IdCliente", "Ciudad", patron_cliente),
    "Producto": catalogo_confiable(trabajo, "IdProducto", "Producto", patron_producto),
    "Categoria": catalogo_confiable(trabajo, "IdProducto", "Categoria", patron_producto),
}
# Un código es confiable cuando existe en el rango observado P001..P012; validamos coherencia contra el catálogo.
validos_producto = {k for k in cat["Producto"] if k in {f"P{i:03d}" for i in range(1,13)}}
validos_cliente = {k for k in cat["Cliente"] if k in {f"C{i:03d}" for i in range(1,21)}}
repetidos = trabajo.IdVenta.duplicated(keep=False)
razones = []
for idx, r in trabajo.iterrows():
    e = []
    if not re.fullmatch(r"V\d{4}", r.IdVenta): e.append("IdVenta sin formato esperado")
    if bool(repetidos.loc[idx]): e.append("IdVenta duplicado: todas las ocurrencias en cuarentena")
    if pd.isna(r.FechaConvertida): e.append("Fecha inválida")
    if pd.isna(r.CantidadConvertida): e.append("Cantidad inválida")
    if pd.isna(r.PrecioConvertido): e.append("PrecioUnitario inválido")
    if r.IdCliente not in validos_cliente: e.append("IdCliente vacío o no reconocido")
    if r.IdProducto not in validos_producto: e.append("IdProducto no reconocido")
    if r.IdCliente in validos_cliente and not cat["Ciudad"].get(r.IdCliente): e.append("Ciudad sin evidencia")
    if r.IdCliente in validos_cliente and not cat["Cliente"].get(r.IdCliente): e.append("NombreCliente sin evidencia")
    if r.IdProducto in validos_producto and not cat["Producto"].get(r.IdProducto): e.append("Producto sin evidencia")
    if r.IdProducto in validos_producto and not cat["Categoria"].get(r.IdProducto): e.append("Categoría sin evidencia")
    if r.IdProducto in validos_producto and r.Producto and firma(r.Producto) != firma(cat["Producto"][r.IdProducto]):
        e.append("Producto incompatible con IdProducto")
    razones.append("; ".join(e))
trabajo["MotivoRechazo"] = razones
cuarentena = trabajo.loc[trabajo.MotivoRechazo.ne("")].copy()
aceptadas = trabajo.loc[trabajo.MotivoRechazo.eq("")].copy()
aceptadas["ClienteCanon"] = aceptadas.IdCliente.map(cat["Cliente"])
aceptadas["CiudadCanon"] = aceptadas.IdCliente.map(cat["Ciudad"])
aceptadas["ProductoCanon"] = aceptadas.IdProducto.map(cat["Producto"])
aceptadas["CategoriaCanon"] = aceptadas.IdProducto.map(cat["Categoria"])
aceptadas["FechaKey"] = aceptadas.FechaConvertida.dt.strftime("%Y%m%d").astype(int)
aceptadas["CantidadOK"] = aceptadas.CantidadConvertida.astype(int)
aceptadas["PrecioOK"] = aceptadas.PrecioConvertido.map(lambda x: x.quantize(Decimal("0.01")))
aceptadas["ImporteOK"] = aceptadas.apply(lambda r: Decimal(r.CantidadOK) * r.PrecioOK, axis=1)
assert len(df) == len(aceptadas) + len(cuarentena)
assert aceptadas.IdVenta.is_unique
print(f"Leídas: {len(df)} | Aceptadas: {len(aceptadas)} | Cuarentena: {len(cuarentena)}")
display(cuarentena[["NumeroFilaCSV", "IdVenta", "MotivoRechazo"]])

Leídas: 500 | Aceptadas: 487 | Cuarentena: 13


,NumeroFilaCSV,IdVenta,MotivoRechazo
111,113,V0112,Cantidad inválida
156,158,V0157,PrecioUnitario inválido
200,202,V0201,IdCliente vacío o no reconocido
242,244,V0243,IdProducto no reconocido
275,277,V0276,IdVenta duplicado: todas las ocurrencias en cu...
276,278,V0276,IdVenta duplicado: todas las ocurrencias en cu...
337,339,V0338,PrecioUnitario inválido
355,357,V0356,Cantidad inválida
389,391,V0390,IdCliente vacío o no reconocido
424,426,V0425,PrecioUnitario inválido


### Interpretación de las anomalías
La política no asume que una fila con precio faltante pueda repararse estimando el precio de otra venta: los precios pueden variar legítimamente. Tampoco asigna automáticamente `P999` usando la descripción porque el mismo identificador aparece con productos distintos. Los valores de ciudad faltantes pueden completarse exclusivamente cuando existe evidencia repetida de la misma clave de cliente.

## Solución 3. Preparar las dimensiones y hechos
Las claves `IDENTITY` de Cliente y Producto serán asignadas por SQL Server. `FechaKey` se genera determinísticamente en formato AAAAMMDD.

In [9]:
dim_cliente = (aceptadas[["IdCliente", "ClienteCanon", "CiudadCanon"]]
    .drop_duplicates(subset=["IdCliente"])
    .rename(columns={"ClienteCanon": "NombreCliente", "CiudadCanon": "Ciudad"})
    .sort_values("IdCliente").reset_index(drop=True))
dim_producto = (aceptadas[["IdProducto", "ProductoCanon", "CategoriaCanon"]]
    .drop_duplicates(subset=["IdProducto"])
    .rename(columns={"ProductoCanon": "NombreProducto", "CategoriaCanon": "Categoria"})
    .sort_values("IdProducto").reset_index(drop=True))
dim_fecha = (aceptadas[["FechaConvertida", "FechaKey"]]
    .drop_duplicates(subset=["FechaKey"])
    .rename(columns={"FechaConvertida": "FechaCompleta"}).sort_values("FechaKey").reset_index(drop=True))
dim_fecha["Dia"] = dim_fecha.FechaCompleta.dt.day
dim_fecha["Mes"] = dim_fecha.FechaCompleta.dt.month
dim_fecha["Trimestre"] = dim_fecha.FechaCompleta.dt.quarter
dim_fecha["Anio"] = dim_fecha.FechaCompleta.dt.year

## Solución 4. Carga transaccional e idempotente
Se utiliza `engine.begin()` para que las inserciones de una corrida constituyan una única transacción. Para el nivel introductorio se emplean sentencias parametrizadas. En escenarios de producción convienen cargas por lotes y políticas SCD claramente definidas.

**Precaución:** las tablas existentes con los mismos códigos y descripciones divergentes detienen la carga: no se modifica en silencio información dimensional previa.

**Secuencia de ejecución:** (a) confirmar la estructura con la prueba inicial; (b) revisar las dimensiones preparadas; (c) habilitar la llamada a `cargar_datamart` para insertar datos en una única transacción; (d) validar conciliación e idempotencia. La carga no se ejecuta automáticamente por seguridad.

In [10]:
def cargar_datamart(conexion_engine, cli, pro, fec, hechos):
    with conexion_engine.begin() as cn:
        for tabla, codigo, attrs, datos in [
            ("DimCliente", "IdCliente", ["NombreCliente", "Ciudad"], cli),
            ("DimProducto", "IdProducto", ["NombreProducto", "Categoria"], pro),
        ]:
            actuales = pd.read_sql_query(text(f"SELECT * FROM dbo.{tabla}"), cn)
            previos = {str(row[codigo]): row for _, row in actuales.iterrows()}
            for r in datos.to_dict("records"):
                key = r[codigo]
                if key in previos:
                    for a in attrs:
                        old = previos[key][a]
                        if (None if pd.isna(old) else str(old)) != (None if pd.isna(r[a]) else str(r[a])):
                            raise ValueError(f"{tabla} {key} tiene divergencia en {a}: revisar catálogo / SCD")
                    continue
                cols = [codigo] + attrs
                sql = text(f"INSERT INTO dbo.{tabla} ({', '.join(cols)}) VALUES ({', '.join(':'+c for c in cols)})")
                cn.execute(sql, {c: r[c] for c in cols})
        fechas_actuales = set(pd.read_sql_query(text("SELECT FechaKey FROM dbo.DimFecha"), cn).FechaKey.astype(int))
        for r in fec.to_dict("records"):
            if int(r["FechaKey"]) not in fechas_actuales:
                cn.execute(text("""INSERT INTO dbo.DimFecha
                    (FechaKey, FechaCompleta, Dia, Mes, Trimestre, Anio)
                    VALUES (:FechaKey, :FechaCompleta, :Dia, :Mes, :Trimestre, :Anio)"""),
                    {**r, "FechaCompleta":r["FechaCompleta"].date(),
                     **{k:int(r[k]) for k in ["FechaKey","Dia","Mes","Trimestre","Anio"]}})
        clientes = dict(pd.read_sql_query(text("SELECT IdCliente, ClienteKey FROM dbo.DimCliente"), cn).values)
        productos = dict(pd.read_sql_query(text("SELECT IdProducto, ProductoKey FROM dbo.DimProducto"), cn).values)
        existentes = set(pd.read_sql_query(text("SELECT IdVentaOrigen FROM dbo.FactVentas"), cn).IdVentaOrigen)
        nuevas = 0
        for r in hechos.itertuples(index=False):
            if r.IdVenta in existentes:
                continue
            if r.IdCliente not in clientes or r.IdProducto not in productos:
                raise ValueError(f"Dimensión no encontrada para venta {r.IdVenta}")
            cn.execute(text("""INSERT INTO dbo.FactVentas
                (IdVentaOrigen, ClienteKey, ProductoKey, FechaKey, Cantidad, PrecioUnitario, ImporteBruto)
                VALUES (:id, :cli, :pro, :fec, :cant, :precio, :importe)"""),
                {"id":r.IdVenta, "cli":int(clientes[r.IdCliente]),
                 "pro":int(productos[r.IdProducto]),"fec":int(r.FechaKey),
                 "cant":int(r.CantidadOK),"precio":r.PrecioOK,"importe":r.ImporteOK})
            existentes.add(r.IdVenta)
            nuevas += 1
    return nuevas

# PRECONDICIÓN: DataMartVentas.sql ya fue ejecutado en SQL Server.
# Descomente para ejecutar la carga REAL:
# insertados = cargar_datamart(engine, dim_cliente, dim_producto, dim_fecha, aceptadas)
# print("Hechos nuevos insertados:", insertados)

## Solución 5. Pruebas de conciliación y consultas
Ejecute la carga descomentando el bloque anterior y luego las consultas siguientes. El escenario de referencia espera `len(aceptadas)` hechos en una base inicialmente vacía. En una base con datos previos se debe comparar por conjunto de `IdVentaOrigen`, no por conteo global.

**Comprobaciones:** comparar identificadores de origen, importes, cantidades, integridad referencial y confirmar que una segunda carga no inserta hechos. El recuento esperado depende de la política de cuarentena documentada; no suponga que los 500 registros son aceptados.

In [ ]:
# Descomente solo tras cargar realmente los datos en SQL Server.
# with engine.connect() as cn:
#     cargadas = pd.read_sql_query(text("SELECT IdVentaOrigen, Cantidad, PrecioUnitario, ImporteBruto FROM dbo.FactVentas"), cn)
#     por_categoria = pd.read_sql_query(text("""
#         SELECT p.Categoria, COUNT(*) AS Lineas, SUM(f.ImporteBruto) AS Total
#         FROM dbo.FactVentas f JOIN dbo.DimProducto p ON p.ProductoKey=f.ProductoKey
#         GROUP BY p.Categoria ORDER BY Total DESC"""), cn)
#     por_ciudad = pd.read_sql_query(text("""
#         SELECT c.Ciudad, COUNT(*) AS Lineas, SUM(f.ImporteBruto) AS Total
#         FROM dbo.FactVentas f JOIN dbo.DimCliente c ON c.ClienteKey=f.ClienteKey
#         GROUP BY c.Ciudad ORDER BY Total DESC"""), cn)
# assert set(aceptadas.IdVenta) <= set(cargadas.IdVentaOrigen)
# assert sum(aceptadas.ImporteOK) == sum(Decimal(str(x)) for x in cargadas.loc[
#     cargadas.IdVentaOrigen.isin(aceptadas.IdVenta), "ImporteBruto"])
# display(por_categoria)
# display(por_ciudad)
# print("Prueba de reejecución: debe insertar cero hechos nuevos")
# assert cargar_datamart(engine, dim_cliente, dim_producto, dim_fecha, aceptadas) == 0

## Evidencias, retroalimentación y alternativas válidas
**Corrección orientativa:** diagnóstico y sustento (25 %), transformación/depuración (25 %), carga dimensional íntegra (25 %), conciliación e idempotencia (20 %), argumentación y trazabilidad (5 %). Ajustable; no sustituye rúbrica oficial.

**Alternativas admisibles:** rechazar todas las claves duplicadas (referencia), aplicar una regla explícita de precedencia *solo si existe evidencia de negocio*, mantener datos en una tabla staging, utilizar `executemany` o `to_sql` para etapas intermedias, y validar con SQL en el servidor. Exigir documentación de impacto en recuentos y totales.

**Errores frecuentes:** convertir nulos a cero, considerar ciudad de cliente ciudad de la venta, asumir secuencia de `IDENTITY`, concatenar SQL sin parámetros, usar `drop_duplicates()` sin analizar diferencias y afirmar que 500 filas deben llegar a hechos.

**Retroalimentación modelo:** «Hallazgo: se cargaron dos ventas con el mismo identificador. Impacto: se sobredimensiona el importe. Recomendación: decidir y justificar una política de resolución/cuarentena, aplicar restricción de unicidad y volver a conciliar».

**Limitación de verificación:** el procesamiento en memoria se puede probar sin servidor, pero la conexión, los permisos, los controladores y la compatibilidad física deben verificarse contra una instancia real de SQL Server.